# Scanpy plotting with GenomeSpy

Keep your Scanpy analysis and replace supported plotting calls with `genome_spy.integrations.scanpy`. This workflow moves from **basic plotting**, to **linked brushing and Python retrieval**, then **customization**. It adapts supported UMAP and dotplot sections of Scanpy’s [core](https://scanpy.scverse.org/en/stable/tutorials/plotting/core.html) and [advanced](https://scanpy.scverse.org/en/stable/tutorials/plotting/advanced.html) plotting tutorials.

We load Scanpy’s real, processed PBMC dataset: 700 cells, 765 genes, annotations, and **UMAP coordinates already stored in `obsm["X_umap"]`**. GenomeSpy reads those coordinates. For your own data, run preprocessing, neighbors, and `sc.tl.umap(adata)` explicitly in Scanpy before plotting.

Install this project with its `scanpy` extra and select that environment as the notebook kernel. The preview targets Scanpy 1.11.5 and currently implements UMAP and plain dotplots.


In [ ]:
import scanpy as sc

import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
pbmc

## 1. Basic plotting

The tutorial’s `sc.pl.umap(pbmc, color=["louvain", "CD79A"])` becomes the call below. The categorical annotation and gene expression panels share the stored embedding but keep independent color scales.

`width` and `height` set each GenomeSpy plot area in logical pixels; axes and legends add space. Hover a cell to inspect its barcode and value. Use the mouse wheel or trackpad over a plot to zoom both axes, and drag to pan. Panels within each chart navigate together.


In [ ]:
chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Summarize marker expression with a dotplot

Use the core tutorial’s marker dictionary and the dataset’s existing `bulk_labels`. Dot color shows the mean expression per group; dot area shows the fraction of cells with expression strictly above zero. This call uses `raw.X` by default, as in Scanpy. Summaries are calculated in Python from selected genes; GenomeSpy renders the chart and its guides.

This is the plain chart first: dendrogram attachment comes in a later milestone. No clustering or differential-expression analysis runs here.


In [ ]:
marker_genes = {
    "B-cell": ["CD79A", "MS4A1"],
    "Dendritic": ["FCER1A", "CST3"],
    "Monocytes": ["FCGR3A"],
    "NK": ["GNLY", "NKG7"],
    "Other": ["IGLL1"],
    "Plasma": ["IGJ"],
    "T-cell": ["CD3D"],
}
dotplot = gsc.pl.dotplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    width=400,
    height=350,
    show=False,
)
dotplot.display(inline=True, controls=False)

## 2. Link a brush across panels

Add a native interval selection to the same plotting call. **Hold Shift and drag a rectangle in the left plot.** Both plots highlight the same cells and the count updates below them. Double-click in the left plot to clear the brush.

Use the mouse wheel or trackpad to zoom both axes, and ordinary dragging to pan. Both panels share their coordinate view. The right plot displays the linked selection; brushing starts in the left plot.


In [ ]:
import genome_spy as gs

brush = gs.selection_interval("cells", encodings=["x", "y"])
chart = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    selection=brush,
    zoom=True,
    width=280,
    height=280,
    show=False,
)
view = chart.widget(parameter_names=[brush.name], inline=True, controls=False)
view

### Retrieve the selected cells in Python

After brushing, run the next cell. The normal Jupyter widget synchronizes rectangle bounds into `view.parameter_values`. `selection_mask` matches them to the stored UMAP coordinates in observation order; barcodes come from `obs_names`.

Run this cell again after changing the brush to refresh the subset. Keep the plotted coordinates unchanged. Highlighting and the count also work in exported interactive HTML; Python retrieval requires a live notebook kernel.


In [ ]:
mask = gsc.selection_mask(pbmc, view.parameter_values.get(brush.name))
selected_ids = pbmc.obs_names[mask]
selected_adata = pbmc[mask].copy()
selected_adata

## 3. Customize the plots

Keep the same dataset and embedding. The following examples cover subsetting, titles, frames, category order, CSS palettes, continuous color bounds, and panel layout. Colors use GenomeSpy schemes and opaque CSS colors; no Matplotlib setup is needed.

### Plot a subset, with a title and no frame

The tutorial’s Louvain groups 0, 3, and 9 are colored by S-phase score. This AnnData view retains its existing coordinates; plotting does not compute a new UMAP.


In [ ]:
subset = pbmc[pbmc.obs["louvain"].isin(["0", "3", "9"])]
chart = gsc.pl.umap(
    subset,
    color="S_score",
    title="S-phase score in selected groups",
    frameon=False,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Set categorical order and colors

Create an ordered annotation explicitly in Python, as in the tutorial. Legend order follows its categorical levels. The same CSS mapping gives each phase a consistent color in both panels.


In [ ]:
pbmc.obs["phase_ordered"] = pbmc.obs["phase"].cat.reorder_categories(
    ["G1", "S", "G2M"], ordered=True
)
chart = gsc.pl.umap(
    pbmc,
    color=["phase", "phase_ordered"],
    title=["Original order", "Ordered phases"],
    palette={"S": "#17becf", "G1": "#bcbd22", "G2M": "#d62728"},
    s=20,
    wspace=0.5,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Use a symmetric continuous color range

The advanced tutorial scores the B-cell markers CD79A and MS4A1 with Scanpy. This explicit analysis adds `B_cell_score` to `obs`; the plotting call only reads it. Use native `redblue` with symmetric numeric bounds.


In [ ]:
sc.tl.score_genes(pbmc, ["CD79A", "MS4A1"], score_name="B_cell_score", random_state=0)
maxabs = abs(pbmc.obs["B_cell_score"]).max()
chart = gsc.pl.umap(
    pbmc,
    color="B_cell_score",
    cmap="redblue",
    s=20,
    vmin=-maxabs,
    vmax=maxabs,
    width=280,
    height=280,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

### Arrange the tutorial’s marker overview

Keep the core tutorial’s marker list, point size, frame setting, four-column layout, and per-panel 99th-percentile upper color limit. The biological labels and `n_counts` come from the packaged dataset. Use horizontal scrolling if the overview is wider than the notebook.


In [ ]:
color_vars = [
    "CD79A",
    "MS4A1",
    "IGJ",
    "CD3D",
    "FCER1A",
    "FCGR3A",
    "n_counts",
    "bulk_labels",
]
chart = gsc.pl.umap(
    pbmc,
    color=color_vars,
    s=50,
    frameon=False,
    ncols=4,
    vmax="p99",
    width=240,
    height=240,
    zoom=True,
    show=False,
)
chart.display(inline=True, controls=False)

Supported UMAP and plain dotplot calls are reproduced here. Matrixplot, dendrograms, heatmaps, and ranked-gene plots follow in the integration roadmap.

> **Data use and provenance**
>
> `scanpy.datasets.pbmc68k_reduced()` loads Scanpy’s packaged, processed subset of 10x Genomics PBMC68k. No dataset copy is bundled here. Category ordering and gene scoring above are explicit notebook operations; the original UMAP coordinates are retained throughout.
